# 第 11 章 インフレと貨幣量: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/11-inflation-and-money-stock.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
money_growth = 5      # 貨幣の伸び率（% / 年）
real_growth = 1       # 実質成長率（% / 年）
velocity_growth = 0   # 流通速度の伸び率（% / 年）
max_inflation = 100   # 回帰に使う、物価上昇率（% / 年）の上限

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 貨幣数量説と、20 か国のパネル

In [ ]:
from lib.chapter11.money import (
    load_money_panel,
    money_inflation_regression,
    quantity_theory_inflation,
)

inflation = quantity_theory_inflation(money_growth, real_growth, velocity_growth)
print("物価上昇率:", inflation)
panel = load_money_panel()
whole = money_inflation_regression(panel)
print("回帰の傾き（全体）:", round(float(whole.coefficients["money_growth"]), 3))
moderate = money_inflation_regression(panel, max_inflation=max_inflation)
print("回帰の傾き（上限より下だけ）:",
      round(float(moderate.coefficients["money_growth"]), 3))

## 図: 貨幣の伸び率と物価上昇率

In [ ]:
import pandas as pd

from lib.chapter03.plot import plot_regression

kept = panel[panel["inflation"] <= max_inflation]
plot_regression(
    pd.Series(kept["money_growth"]),
    pd.Series(kept["inflation"]),
    xlabel="Money growth (% per year)",
    ylabel="Inflation (% per year)",
    label="Country-years",
)

## 試してみよう

- `real_growth = 3` にすると、物価上昇率は 2 です（貨幣が 5 % で、実質成長が 3 %）
- `money_growth = 10`、`real_growth = 0` にすると、物価上昇率は 10 です
- `velocity_growth = 2` にすると、物価上昇率は 6 です（流通速度が上がると、
  物価も上がります）
- `max_inflation = 1000` にすると、ハイパーインフレも含むので、
  回帰の傾きは全体と同じ 1.029 です